# ArtInsight - Dataset Loader for Baseline Training

This notebook uses the output generated by the VIA inspector and assumes that the following folders already exist:

- original JPG images
- generated PNG masks

Objectives:
- verify that each image matches its corresponding mask
- build a list of training-ready samples
- visualize sample examples
- create a basic PyTorch `Dataset`
- confirm that a `DataLoader` works correctly

This notebook does not perform training yet. Its purpose is to prepare the dataset for the baseline.

## 0. Imports and Configuration

In [ ]:
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict
import random

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# PyTorch
import torch
from torch.utils.data import Dataset, DataLoader

# Adjust these paths to your environment
ROOT = Path('Dataset')
INSPECTOR_OUTPUT = Path('outputs') / 'via_dataset_inspector'  # extracted zip folder

MASKS_ROOT = INSPECTOR_OUTPUT / 'masks'
OVERLAYS_ROOT = INSPECTOR_OUTPUT / 'overlays'

print('ROOT =', ROOT)
print('MASKS_ROOT =', MASKS_ROOT)
print('OVERLAYS_ROOT =', OVERLAYS_ROOT)
print('ROOT exists:', ROOT.exists())
print('MASKS_ROOT exists:', MASKS_ROOT.exists())
print('OVERLAYS_ROOT exists:', OVERLAYS_ROOT.exists())

ROOT = Dataset
MASKS_ROOT = Dataset\outputs\via_dataset_inspector\masks
OVERLAYS_ROOT = Dataset\outputs\via_dataset_inspector\overlays
ROOT exists: True
MASKS_ROOT exists: False
OVERLAYS_ROOT exists: False


## 1. Check Available Structure

In [11]:
for base in [ROOT, MASKS_ROOT, OVERLAYS_ROOT]:
    print('\n===', base, '===')
    if base.exists():
        for p in sorted(base.rglob('*')):
            if p.is_file():
                print(p.relative_to(base))
    else:
        print('Not found')


=== Dataset ===
.DS_Store
lpl\.DS_Store
lpl\test\lpl_test.json
lpl\test\Pintura_0.10.jpg
lpl\test\Pintura_0.16.jpg
lpl\train\lpl_train.json
lpl\train\Pintura_0.12.jpg
lpl\train\Pintura_0.15.jpg
lpl\train\Pintura_0.18.jpg
lpl\train\Pintura_0.19.jpg
lpl\train\Pintura_0.3.jpg
lpl\train\Pintura_0.6.jpg
model.h5
readme.txt
stucco\.DS_Store
stucco\test\Pintura_0.1.jpg
stucco\test\Pintura_0.11.jpg
stucco\test\Pintura_0.20.jpg
stucco\test\Pintura_0.8.jpg
stucco\test\stucco_test.json
stucco\train\Pintura_0.13.jpg
stucco\train\Pintura_0.14.jpg
stucco\train\Pintura_0.17.jpg
stucco\train\Pintura_0.2.jpg
stucco\train\Pintura_0.4.jpg
stucco\train\Pintura_0.5.jpg
stucco\train\Pintura_0.7.jpg
stucco\train\Pintura_0.9.jpg
stucco\train\stucco_train.json

=== Dataset\outputs\via_dataset_inspector\masks ===
Not found

=== Dataset\outputs\via_dataset_inspector\overlays ===
Not found


## 2. Define Expected Splits

In [12]:
SPLITS = [
    ('lpl', 'train'),
    ('lpl', 'test'),
    ('stucco', 'train'),
    ('stucco', 'test'),
]

SPLITS

[('lpl', 'train'), ('lpl', 'test'), ('stucco', 'train'), ('stucco', 'test')]

## 3. Build Image-to-Mask Inventory

In [13]:
@dataclass
class Sample:
    task: str
    split: str
    image_path: Path
    mask_path: Path
    filename: str
    class_id: int


def class_id_from_task(task: str) -> int:
    mapping = {'stucco': 0, 'lpl': 1}
    if task not in mapping:
        raise ValueError(f'Unknown task: {task}')
    return mapping[task]


def find_image_path(dataset_root: Path, task: str, split: str, filename: str) -> Path:
    candidate = dataset_root / task / split / filename
    if candidate.exists():
        return candidate
    raise FileNotFoundError(f'Image not found: {candidate}')


samples: List[Sample] = []
missing_pairs = []

for task, split in SPLITS:
    mask_dir = MASKS_ROOT / task / split
    if not mask_dir.exists():
        print('Mask directory not found:', mask_dir)
        continue

    for mask_path in sorted(mask_dir.glob('*.png')):
        filename = mask_path.stem + '.jpg'
        try:
            image_path = find_image_path(ROOT, task, split, filename)
            samples.append(
                Sample(
                    task=task,
                    split=split,
                    image_path=image_path,
                    mask_path=mask_path,
                    filename=filename,
                    class_id=class_id_from_task(task),
                )
            )
        except FileNotFoundError:
            missing_pairs.append((task, split, filename, str(mask_path)))

print('Total samples found:', len(samples))
print('Missing pairs:', len(missing_pairs))
missing_pairs[:10]

Mask directory not found: Dataset\outputs\via_dataset_inspector\masks\lpl\train
Mask directory not found: Dataset\outputs\via_dataset_inspector\masks\lpl\test
Mask directory not found: Dataset\outputs\via_dataset_inspector\masks\stucco\train
Mask directory not found: Dataset\outputs\via_dataset_inspector\masks\stucco\test
Total samples found: 0
Missing pairs: 0


[]

## 4. Summary by Split

In [14]:
from collections import Counter

split_counter = Counter((s.task, s.split) for s in samples)
class_counter = Counter(s.class_id for s in samples)

print('Summary by split:')
for k, v in sorted(split_counter.items()):
    print(k, '->', v)

print('\nSummary by class_id:')
print(class_counter)

Summary by split:

Summary by class_id:
Counter()


## 5. Verify Image and Mask Dimensions

In [15]:
size_mismatches = []

for s in samples:
    img = Image.open(s.image_path)
    mask = Image.open(s.mask_path)

    if img.size != mask.size:
        size_mismatches.append((s.filename, img.size, mask.size, s.task, s.split))

print('Image/mask size mismatches:', len(size_mismatches))
size_mismatches[:10]

Image/mask size mismatches: 0


[]

## 6. Visualize Random Samples

In [16]:
def show_sample(sample: Sample):
    img = np.array(Image.open(sample.image_path).convert('RGB'))
    mask = np.array(Image.open(sample.mask_path).convert('L'))

    plt.figure(figsize=(14, 6))
    plt.subplot(1, 2, 1)
    plt.imshow(img)
    plt.title(f'Image | {sample.task}/{sample.split} | {sample.filename}')
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.imshow(mask, cmap='gray')
    plt.title('PNG Mask')
    plt.axis('off')
    plt.show()

random.seed(42)
for s in random.sample(samples, min(4, len(samples))):
    show_sample(s)

## 7. Quick Overlay Using the Mask

In [17]:
def show_overlay_from_mask(sample: Sample, alpha: float = 0.35):
    img = np.array(Image.open(sample.image_path).convert('RGB')).astype(np.float32)
    mask = np.array(Image.open(sample.mask_path).convert('L'))
    mask_bin = (mask > 0).astype(np.float32)

    overlay = img.copy()
    overlay[..., 0] = np.where(mask_bin > 0, img[..., 0] * (1 - alpha) + 255 * alpha, img[..., 0])

    plt.figure(figsize=(8, 8))
    plt.imshow(overlay.astype(np.uint8))
    plt.title(f'Mask Overlay | {sample.task}/{sample.split} | {sample.filename}')
    plt.axis('off')
    plt.show()

for s in random.sample(samples, min(2, len(samples))):
    show_overlay_from_mask(s)

## 8. Basic Mask Statistics

In [ ]:
mask_stats = []

for s in samples:
    mask = np.array(Image.open(s.mask_path).convert('L'))
    positive = int((mask > 0).sum())
    total = mask.size
    ratio = positive / total if total > 0 else 0.0
    mask_stats.append((s.filename, s.task, s.split, positive, total, ratio))

print('Top 10 masks with the most positive pixels:')
for row in sorted(mask_stats, key=lambda x: x[3], reverse=True)[:10]:
    print(row)

print('\nTop 10 masks with the fewest positive pixels:')
for row in sorted(mask_stats, key=lambda x: x[3])[:10]:
    print(row)

## 9. Basic PyTorch Dataset

In [ ]:
class ArtInsightSegmentationDataset(Dataset):
    def __init__(self, samples: List[Sample], resize=None):
        self.samples = samples
        self.resize = resize  # tuple (W, H) or None

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]

        img = Image.open(s.image_path).convert('RGB')
        mask = Image.open(s.mask_path).convert('L')

        if self.resize is not None:
            img = img.resize(self.resize, Image.BILINEAR)
            mask = mask.resize(self.resize, Image.NEAREST)

        img_np = np.array(img, dtype=np.float32) / 255.0
        mask_np = (np.array(mask, dtype=np.uint8) > 0).astype(np.float32)

        img_tensor = torch.from_numpy(img_np).permute(2, 0, 1)  # C,H,W
        mask_tensor = torch.from_numpy(mask_np).unsqueeze(0)     # 1,H,W

        target = {
            'mask': mask_tensor,
            'class_id': torch.tensor(s.class_id, dtype=torch.long),
            'task': s.task,
            'split': s.split,
            'filename': s.filename,
        }

        return img_tensor, target

dataset = ArtInsightSegmentationDataset(samples=samples, resize=(512, 512))
print('Dataset length:', len(dataset))

## 10. Test a Dataset Sample

In [ ]:
img_tensor, target = dataset[0]

print('img_tensor shape:', img_tensor.shape)
print('mask_tensor shape:', target['mask'].shape)
print('class_id:', target['class_id'])
print('task:', target['task'])
print('split:', target['split'])
print('filename:', target['filename'])

## 11. Visualize a Resized Sample

In [ ]:
img_show = img_tensor.permute(1, 2, 0).numpy()
mask_show = target['mask'].squeeze(0).numpy()

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.imshow(img_show)
plt.title('Tensorized Image')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(mask_show, cmap='gray')
plt.title('Tensorized Mask')
plt.axis('off')
plt.show()

## 12. Minimal DataLoader

In [ ]:
def simple_collate(batch):
    images = torch.stack([item[0] for item in batch], dim=0)
    masks = torch.stack([item[1]['mask'] for item in batch], dim=0)
    class_ids = torch.stack([item[1]['class_id'] for item in batch], dim=0)
    meta = [{
        'task': item[1]['task'],
        'split': item[1]['split'],
        'filename': item[1]['filename'],
    } for item in batch]
    return images, masks, class_ids, meta

loader = DataLoader(dataset, batch_size=2, shuffle=True, num_workers=0, collate_fn=simple_collate)

images, masks, class_ids, meta = next(iter(loader))
print('images:', images.shape)
print('masks:', masks.shape)
print('class_ids:', class_ids.shape)
print('meta[0]:', meta[0])

## 13. Separate Train and Test for the Baseline

In [ ]:
train_samples = [s for s in samples if s.split == 'train']
test_samples = [s for s in samples if s.split == 'test']

print('train_samples:', len(train_samples))
print('test_samples:', len(test_samples))

train_dataset = ArtInsightSegmentationDataset(train_samples, resize=(512, 512))
test_dataset = ArtInsightSegmentationDataset(test_samples, resize=(512, 512))

print('train_dataset:', len(train_dataset))
print('test_dataset:', len(test_dataset))

## 14. Save Inventory to CSV

In [ ]:
import csv

out_csv = ROOT / 'artinsight_samples_inventory.csv'

with open(out_csv, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['task', 'split', 'filename', 'class_id', 'image_path', 'mask_path'])
    for s in samples:
        writer.writerow([s.task, s.split, s.filename, s.class_id, str(s.image_path), str(s.mask_path)])

print('Inventory saved to:', out_csv)

## 15. Final Checklist

- Do the image and mask have the same dimensions?
- Do the masks look visually consistent?
- Does the dataset return the correct tensors?
- Does the dataloader run without errors?
- Are train and test properly separated?

If all of the above checks pass, the next step is to build a segmentation baseline.